# Sadeghi et al. (2019): auditory model classifier claims (CRCNS bst-2)

Paper: Sadeghi M, Zhai X, Stevenson IH, Escabi MA (2019). A neural ensemble correlation code for
sound category identification. PLoS Biol 17(10):e3000449, doi:10.1371/journal.pbio.3000449. The
claims file (claims/sadeghi-2019.yaml) was written from the version of record (PMC6788721) with
its supplement.

Dataset: CRCNS bst-2 (doi:10.6080/K03X84V3), bst-2/live/data.tar.gz in the public bucket
crcnsarchive, `crcns:bst-2/unversioned` (CRCNS does not version data sets; the paper cites the DOI
only, and this is the bucket's only copy). The model sounds are
data/Sounds/ModelSounds/<category>/Sound_<n>.wav. The deposit's code (data/Code) has no classifier
code, so the classifier follows the Methods.

This notebook tests the model classification claims; the model correlation claims are in
reproduce_sadeghi_2019_model.ipynb and the neural claims in reproduce_sadeghi_2019_neural.ipynb.

| Claim | Tested here |
|---|---|
| fig9a-optimal-resolution | tested |
| fig9b-similar-maximum | tested |
| fig9b-accumulation-rate | tested |
| fig9c-time-averaged-worse | tested |
| fig9e-category-specific | tested |
| fig9-spectrum-classifier-lower | tested |
| fig10-vocalization-background | tested |

Method (Methods, "Dimensionality reduction and distribution model", "Bayesian classifier",
"Cross validation", "Optimal temporal resolution and integration time", "Spectrum-based
classifier"; model.py says where each parameter comes from): the cochleogram and short-term
correlations of reproduce_sadeghi_2019_model.ipynb at the ten resolutions tau_W = 25 to 566 ms;
spectral features c_kl(t, 0) without the diagonal, temporal features c_kk(t, tau) for tau from 1
ms to tau_W / 2 with time points and channels as observations, spectro-temporal features both side
by side, the spectrum feature the windowed mean envelope of each channel; PCA; one diagonal GMM
per category; leave one sound out; N consecutive non-overlapping windows from the start of the
test sound, N in half octave steps; log-likelihoods summed; equal priors. Readings of the claims
file: (6) the window spacing, (a) tau_W, (b) the full Kaiser window length; (7) PCs and N_c, (a)
refitted at each tau_W (PCs explaining 90% of the variance; N_c is the reported count, and the
cross-validated choice of N_c is run at each classifier's optimum), (b) the reported counts (26,
8 and 87 PCs; 8, 5 and 13 components) at every tau_W. Chance is 1/13 = 7.69%.

In [ ]:
import time, resource
import numpy as np
from scipy import stats, optimize
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import sadeghi as S
import model as M

threadpool_limits(1)  # many small fits run in parallel processes
T_START = time.time()
SEED = {"pca": 11, "loso": 21, "avg": 31, "two": 41, "nc": 51, "fold_pca": 61}  # standard choices
CHANCE = 100 / 13

## Loading

The 195 model sounds are read from the tar stream (the first members after the paradigm sounds;
the stream is closed once they are read) and each is passed through the cochlear model.

In [ ]:
raw = {}
for name, b in S.stream_members(lambda n: "/Sounds/ModelSounds/" in n and n.endswith(".wav")):
    raw[name] = b
    if len(raw) == 195:
        break
KEYS = sorted((n.split("/")[-2], int(n.split("_")[-1][:-4])) for n in raw)
CATS = sorted({k[0] for k in KEYS})
LAB = np.array([CATS.index(k[0]) for k in KEYS])
by_key = {(n.split("/")[-2], int(n.split("_")[-1][:-4])): b for n, b in raw.items()}

WAV = [M.read_wav(by_key[k]) for k in KEYS]
COCH = [M.cochleogram(x, fs).astype(np.float32) for fs, x in WAV]
del raw, by_key
print(f"read and filtered {len(COCH)} sounds in {(time.time() - T_START) / 60:.1f} min")

In [ ]:
print("categories:", len(CATS), "; sounds per category:", np.bincount(LAB).tolist())
print("sampling rates:", sorted({w[0] for w in WAV}), "; samples per sound:", sorted({len(w[1]) for w in WAV}))
print("cochleogram shape (channels, ms):", sorted({c.shape for c in COCH}))
print("envelope range over all sounds:", round(float(min(c.min() for c in COCH)), 3), "to", round(float(max(c.max() for c in COCH)), 3),
      "(the sound is normalised to unit SD before filtering)")
F0 = M.features(COCH[0].astype(np.float64), 100)
print("features at tau_W = 100 ms:", {k: v.shape for k, v in F0.items() if hasattr(v, 'shape')}, "; Kaiser length", F0["klen"], "ms")
print("spectral feature range", round(float(F0['spec'].min()), 3), "to", round(float(F0['spec'].max()), 3),
      "; temporal", round(float(F0['temp'].min()), 3), "to", round(float(F0['temp'].max()), 3), "(correlations, within -1 to 1)")
print("resolutions tau_W (ms):", np.round(M.RES_MS, 1))
print("Kaiser window length (ms) at each:", [M.kaiser_for(t / 1000)[0].size for t in M.RES_MS])
del WAV, F0

## Classifiers at every resolution

For each tau_W: features of the 195 sounds; PCA of each feature set fitted on the training
observations of all sounds (an assumption: PCA is unsupervised and the held-out sound is 1 of 195;
a variant below refits it in every fold); leave-one-sound-out GMM classifiers under reading (7b)
and, where the PC count for 90% of the variance differs from the reported one, (7a); test windows
under readings (6a) and (6b). The spectrum classifier has no reported counts; it uses the PCs
explaining 90% and 8 components (the spectral count) under both readings.

In [ ]:
KINDS = ["spec", "temp", "st", "spectrum"]
LL, NPC90, CUM, NWIN, SPACING, REFITTED, ZS, TIDX = {}, {}, {}, {}, {}, {}, {}, {}
t0 = time.time()
for tw in M.RES_MS:
    Fs = Parallel(n_jobs=4)(delayed(M.features)(c.astype(np.float64), tw) for c in COCH)
    ia, ib = M.test_index(Fs[0], tw), M.test_index(Fs[0], Fs[0]["klen"])
    NWIN[tw] = {"a": len(ia), "b": len(ib)}
    TIDX[tw] = {"a": ia, "b": ib}
    SPACING[tw] = {"a": Fs[0]["step"] * int(np.ceil(tw / Fs[0]["step"] - 1e-9)) / 1000,
                   "b": Fs[0]["step"] * int(np.ceil(Fs[0]["klen"] / Fs[0]["step"] - 1e-9)) / 1000}
    for kind in KINDS:
        mean, comps, cum = M.fit_pca(Fs, kind, np.random.default_rng(SEED["pca"]))
        Z = [M.project(F, kind, mean, comps) for F in Fs]
        n90 = int(min(np.searchsorted(cum, 0.9) + 1, len(cum)))
        NPC90[(kind, tw)], CUM[(kind, tw)] = n90, cum
        tests = [{"a": z[ia], "b": z[ib]} for z in Z]
        rep = M.REPORTED.get(kind, (n90, 8))
        runs = {"7b": rep}
        if n90 != rep[0]:
            runs["7a"] = (n90, rep[1])
        for r7, (npc, nc) in runs.items():
            ll, full, refit = M.loso(Z, LAB, npc, nc, tests, SEED["loso"])
            REFITTED[(kind, r7, tw)] = sum(g.refitted_ for g in full + refit)
            for sp in "ab":
                LL[(kind, r7, sp, tw)] = ll[sp]
        if "7a" not in runs:
            for sp in "ab":
                LL[(kind, "7a", sp, tw)] = LL[(kind, "7b", sp, tw)]
        kk = max(v[0] for v in runs.values())
        ZS[(kind, tw)] = [np.ascontiguousarray(z[:, :, :kk]) for z in Z]  # kept for fig9c and fig10
        del Z, tests
    del Fs
    print(f"tau_W {tw:6.1f} ms: windows (6a) {NWIN[tw]['a']}, (6b) {NWIN[tw]['b']}; PCs for 90%: "
          + ", ".join(f"{k} {NPC90[(k, tw)]}" for k in KINDS) + f" | {(time.time() - t0) / 60:.1f} min,"
          f" peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB", flush=True)
print("GMM fits refitted with the larger variance floor:", sum(REFITTED.values()), "of", len(REFITTED) * 208)

In [ ]:
READ = [(r7, sp) for r7 in ("7b", "7a") for sp in "ab"]
ACC = {}  # (kind, r7, sp, tw) -> (195 sounds, durations) 0/1
NS = {}
for (kind, r7, sp, tw), ll in LL.items():
    ns = M.n_steps(NWIN[tw][sp])
    NS[(sp, tw)] = ns
    ACC[(kind, r7, sp, tw)] = M.accuracy_curve(ll, LAB, np.arange(13), ns)
def at10(kind, r7, sp, tw):
    return 100 * ACC[(kind, r7, sp, tw)][:, -1].mean()
print("accuracy (%) at the full sound duration (the largest N), by resolution:")
for r7, sp in READ:
    print(f"reading ({r7})({sp}):")
    for kind in KINDS:
        print(f"   {kind:9s}", " ".join(f"{at10(kind, r7, sp, tw):5.1f}" for tw in M.RES_MS))
print("   tau_W    ", " ".join(f"{tw:5.0f}" for tw in M.RES_MS))
print("full duration (s) of the largest N: (6a)", [round(NS[('a', tw)][-1] * SPACING[tw]['a'], 2) for tw in M.RES_MS])
print("                                    (6b)", [round(NS[('b', tw)][-1] * SPACING[tw]['b'], 2) for tw in M.RES_MS])

## fig9a-optimal-resolution

> In the 13-category model task, the spectral and temporal correlation classifiers perform best
> at a window resolution of 141 ms and the spectro-temporal classifier at 100 ms, and all perform
> above chance (7.69%) at every resolution.

Optimum: the resolution with the highest accuracy at the full duration (ties go to the first).
t test of the 195 per-sound outcomes against 7.69%, Bonferroni over the 10 resolutions; the 13
category accuracies are a variant.

In [ ]:
OPT = {}
for r7, sp in READ:
    for kind in KINDS:
        accs = [at10(kind, r7, sp, tw) for tw in M.RES_MS]
        OPT[(kind, r7, sp)] = M.RES_MS[int(np.argmax(accs))]
    worst = []
    for kind in ("spec", "temp", "st"):
        for tw in M.RES_MS:
            x = 100 * ACC[(kind, r7, sp, tw)][:, -1]
            p = stats.ttest_1samp(x, CHANCE, alternative="greater").pvalue * 10
            cat = np.array([x[LAB == c].mean() for c in range(13)])
            pc = stats.ttest_1samp(cat, CHANCE, alternative="greater").pvalue * 10
            worst.append((min(p, 1), min(pc, 1), x.mean(), kind, tw))
    pmax = max(worst)[0]; pcmax = max(w[1] for w in worst); lo = min(worst, key=lambda w: w[2])
    o = {k: OPT[(k, r7, sp)] for k in KINDS}
    ok = (abs(o["spec"] - 141.4) < 1 and abs(o["temp"] - 141.4) < 1 and abs(o["st"] - 100) < 1)
    step = lambda a, b: abs(np.log2(a / b)) <= 0.5 + 1e-9
    near = step(o["spec"], 141.4) and step(o["temp"], 141.4) and step(o["st"], 100)
    print(f"({r7})({sp}): optimum spectral {o['spec']:.1f}, temporal {o['temp']:.1f}, spectro-temporal {o['st']:.1f}, spectrum {o['spectrum']:.1f} ms"
          f" | on the reported grid points: {ok} | within one grid step: {near}")
    print(f"      lowest correlation-classifier accuracy {lo[2]:.1f}% ({lo[3]} at {lo[4]:.0f} ms); largest Bonferroni p over sounds {pmax:.1e},"
          f" over category means {pcmax:.1e}; every resolution above chance at p < 0.01: {pmax < 0.01}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, (r7, sp) in zip(ax, (("7b", "a"), ("7b", "b"))):
    for kind in KINDS:
        a.plot(M.RES_MS, [at10(kind, r7, sp, tw) for tw in M.RES_MS], "o-", label=kind)
    a.axhline(CHANCE, color="k", lw=0.5); a.set_xscale("log"); a.set_xlabel("tau_W (ms)"); a.set_title(f"({r7})({sp}), full duration")
ax[0].set_ylabel("% correct"); ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()
print("plotted range:", round(min(at10(k, '7b', s, tw) for k in KINDS for s in 'ab' for tw in M.RES_MS), 1), "to",
      round(max(at10(k, '7b', s, tw) for k in KINDS for s in 'ab' for tw in M.RES_MS), 1), "%")

**Result: under readings (6b)(7b) the spectral and temporal classifiers are best at 141 ms and the spectro-temporal classifier at 100 ms, as reported; under the other readings the optima move to neighbouring or further grid points (spectral 283 ms under (6a)(7b), spectro-temporal 283 ms under (7a)), as the accuracy curves over resolution are flat. Every correlation classifier is above 7.69% at every resolution under every reading (lowest 58.5%, largest Bonferroni p 1.3e-31 over sounds).**

## fig9b-similar-maximum

> At about 10 s the three correlation classifiers reach similar performance (spectro-temporal 84%,
> spectral 83%, temporal 81%), with no significant pairwise differences.

Each classifier at its own optimum (fig9a), full duration; two-sample t tests of the 195
per-sound outcomes for each pair, with the 13 category accuracies as a variant.

In [ ]:
EXP9B = {"st": (84, 8.4), "spec": (83, 8.3), "temp": (81, 8.1)}
for r7, sp in READ:
    x = {k: 100 * ACC[(k, r7, sp, OPT[(k, r7, sp)])][:, -1] for k in EXP9B}
    hits = sum(abs(x[k].mean() - e) <= t for k, (e, t) in EXP9B.items())
    line = []
    pmin = 1
    for a, b in (("spec", "temp"), ("spec", "st"), ("temp", "st")):
        p = stats.ttest_ind(x[a], x[b]).pvalue
        ca = [x[a][LAB == c].mean() for c in range(13)]; cb = [x[b][LAB == c].mean() for c in range(13)]
        pc = stats.ttest_ind(ca, cb).pvalue
        pmin = min(pmin, p)
        line.append(f"{a}-{b} {x[a].mean() - x[b].mean():+.1f} points, p {p:.2f} (categories p {pc:.2f})")
    print(f"({r7})({sp}): spectro-temporal {x['st'].mean():.1f}, spectral {x['spec'].mean():.1f}, temporal {x['temp'].mean():.1f}%"
          f" | in tolerance {hits}/3 | " + "; ".join(line) + f" | no pair differs at 0.05: {pmin >= 0.05}")

**Result: at their optima and the full duration the three classifiers are within tolerance under every reading ((6b)(7b) spectro-temporal 81.5, spectral 82.1, temporal 80.5%) and no pair differs under the (7b) readings (p 0.43 to 0.90); under (6a)(7a) the spectro-temporal classifier reaches 87.7% and differs from the temporal one (p = 0.04 over sounds).**

## fig9b-accumulation-rate

> The spectral classifier reaches 90% of its maximum faster than the temporal one (1.7 against
> 3.0 s), and the spectro-temporal classifier fastest (1.2 s).

Rise time to 90% of the accuracy at the full duration, on the curves of fig9b: (a) linear
interpolation between the sampled durations, (b) the first sampled duration reaching it, (c) an
exponential fit P(t) = P_inf - (P_inf - P_0) exp(-t / tau). Duration is N times the window spacing.

In [ ]:
EXPR = {"spec": (1.7, 0.17), "temp": (3.0, 0.3), "st": (1.2, 0.12)}
def rise(t, y):
    thr = 0.9 * y[-1]
    i = int(np.argmax(y >= thr))
    lin = t[0] if i == 0 else t[i - 1] + (t[i] - t[i - 1]) * (thr - y[i - 1]) / (y[i] - y[i - 1])
    try:
        f = lambda tt, pinf, p0, tau: pinf - (pinf - p0) * np.exp(-tt / tau)
        (pinf, p0, tau), _ = optimize.curve_fit(f, t, y, p0=(y[-1], y[0], 1.0), maxfev=20000)
        tt = np.linspace(0, t[-1], 100001)
        yy = f(tt, pinf, p0, tau)
        ex = float(tt[np.argmax(yy >= 0.9 * f(t[-1], pinf, p0, tau))])
    except RuntimeError:
        ex = np.nan
    return lin, t[i], ex
CURVES = {}
for r7, sp in READ:
    res = {}
    for k in EXPR:
        tw = OPT[(k, r7, sp)]
        t = np.array(NS[(sp, tw)]) * SPACING[tw][sp]
        y = 100 * ACC[(k, r7, sp, tw)].mean(0)
        CURVES[(k, r7, sp)] = (t, y)
        res[k] = rise(t, y)
    for j, lab in enumerate(("(a) interpolated", "(b) first sampled", "(c) exponential fit")):
        v = {k: res[k][j] for k in EXPR}
        hits = sum(abs(v[k] - e) <= tol for k, (e, tol) in EXPR.items())
        order = v["st"] < v["spec"] < v["temp"]
        print(f"({r7})({sp}) {lab}: spectro-temporal {v['st']:.2f}, spectral {v['spec']:.2f}, temporal {v['temp']:.2f} s"
              f" | in tolerance {hits}/3 | order st < spectral < temporal: {order}")
for k in EXPR:
    t, y = CURVES[(k, "7b", "a")]
    print(f"curve (7b)(a) {k} at {OPT[(k, '7b', 'a')]:.0f} ms: " + ", ".join(f"{a:.2f} s {b:.1f}%" for a, b in zip(t, y)))
fig, ax = plt.subplots(figsize=(4.5, 3))
for k in list(EXPR) + ["spectrum"]:
    tw = OPT[(k, "7b", "a")]
    ax.plot(np.array(NS[("a", tw)]) * SPACING[tw]["a"], 100 * ACC[(k, "7b", "a", tw)].mean(0), "o-", ms=3, label=k)
ax.set_xscale("log"); ax.set_xlabel("duration (s)"); ax.set_ylabel("% correct"); ax.legend(fontsize=7); ax.set_title("(7b)(a), each at its optimum")
plt.tight_layout(); plt.show()

**Result: no reading puts the three rise times within tolerance; under (6b)(7b) with interpolation the spectral classifier needs 1.92 s, the temporal 2.67 s and the spectro-temporal 1.23 s. The order spectro-temporal < spectral < temporal holds under the (7b) readings and (6a)(7a) for every measure, not under (6b)(7a).**

## fig9c-time-averaged-worse

> Ignoring nonstationary structure by averaging the spectro-temporal statistics over time reduces
> category identification performance by about 25%.

At the spectro-temporal optimum, full duration. Reading (a): one time-averaged spectro-temporal
vector per training sound (the mean of its short-term PC scores, the same PCs as the short-term
classifier; PCA is linear, so this is the projection of the time-averaged correlation), one
diagonal Gaussian per category (N_c = 1, as a category has only 14 or 15 such vectors), leave one
sound out; the test vector is the average over the first N windows.
Reading (b): the short-term category models of the main classifier, and the test feature is the
running average of the short-term correlation over the first N windows, scored as one
observation. Reduction (c) in points, (d) relative to the short-term accuracy.

In [ ]:
def time_avg_classifier(r7, sp):
    tw = OPT[("st", r7, sp)]
    idx = TIDX[tw][sp]
    ns = NS[(sp, tw)]
    npc, nc = M.REPORTED["st"] if r7 == "7b" else (NPC90[("st", tw)], M.REPORTED["st"][1])
    Z = ZS[("st", tw)]
    # (b): running averages of the test windows, scored under the short-term models
    tests = [{"run": np.array([z[idx][:n].mean(0) for n in ns])} for z in Z]
    llb, _, _ = M.loso(Z, LAB, npc, nc, tests, SEED["loso"])
    acc_b = np.array([[int(np.argmax(L[i])) == LAB[j] for i in range(len(ns))] for j, L in enumerate(llb["run"])], float)
    # (a): one time-averaged vector per sound (the mean of its short-term PC scores, which is the
    # projection of its time-averaged correlation), one diagonal Gaussian per category
    X = np.array([z[:, 0, :npc].mean(0) for z in Z], np.float64)
    Xw = [np.array([z[idx][:n, 0, :npc].mean(0) for n in ns], np.float64) for z in Z]
    acc_a = np.zeros((195, len(ns)))
    for j in range(195):
        tr = np.arange(195) != j
        ll = np.zeros((len(ns), 13))
        for c in range(13):
            pc = X[tr & (LAB == c)]
            m, v = pc.mean(0), pc.var(0) + 1e-6 * X[tr].var(0).max()
            ll[:, c] = -0.5 * (((Xw[j] - m) ** 2) / v + np.log(2 * np.pi * v)).sum(1)
        acc_a[j] = ll.argmax(1) == LAB[j]
    return tw, ns, acc_a, acc_b, npc
AVG = {}
for r7, sp in READ:
    tw = OPT[("st", r7, sp)]
    if r7 == "7a" and OPT[("st", "7b", sp)] == tw and NPC90[("st", tw)] == M.REPORTED["st"][0]:
        AVG[(r7, sp)] = AVG[("7b", sp)]  # the same classifier
    else:
        AVG[(r7, sp)] = time_avg_classifier(r7, sp)
for (r7, sp), (tw, ns, acc_a, acc_b, k) in AVG.items():
    st = at10("st", r7, sp, tw)
    for lab, acc in (("(a) time-averaged models", acc_a), ("(b) running average", acc_b)):
        v = 100 * acc[:, -1].mean()
        d_pts, d_rel = st - v, 100 * (st - v) / st
        print(f"({r7})({sp}) {lab} at {tw:.0f} ms, {k} PCs: {v:.1f}% against short-term {st:.1f}% | reduction (c) {d_pts:.1f} points"
              f" (in 25 +/- 2.5: {abs(d_pts - 25) <= 2.5}), (d) {d_rel:.1f}% (in tolerance: {abs(d_rel - 25) <= 2.5}) | at least 10 points lower: {d_pts >= 10}")
    print(f"      curves (a): " + ", ".join(f"N={n} {100 * m:.1f}" for n, m in zip(ns, acc_a.mean(0)))
          + " | (b): " + ", ".join(f"N={n} {100 * m:.1f}" for n, m in zip(ns, acc_b.mean(0))))

**Result: time-averaged category models (reading a) are only 5.1 to 5.6 points below the short-term classifier; scoring the running average of the test sound under the short-term models (reading b) is 15.4 to 36.9 points lower. No reading gives 25 +/- 2.5; the reduction is at least 10 points under reading (b) only.**

## fig9e-category-specific

> Some categories are identified much better by one feature set; waves by spectral correlations
> (100% against 33%) and wind by temporal ones (47% against 80%).

Per-category accuracy over its 15 sounds, each classifier at its optimum, full duration.

In [ ]:
EXP9E = [("Wave", "spec", 100, 10), ("Wave", "temp", 33, 6.7), ("Wind", "spec", 47, 6.7), ("Wind", "temp", 80, 8)]
for r7, sp in READ:
    pc = {k: [100 * ACC[(k, r7, sp, OPT[(k, r7, sp)])][LAB == c, -1].mean() for c in range(13)] for k in ("spec", "temp", "st")}
    v = [pc[k][CATS.index(c)] for c, k, _, _ in EXP9E]
    hits = sum(abs(x - e) <= t + 1e-9 for x, (_, _, e, t) in zip(v, EXP9E))
    part = (v[0] - v[1] >= 13.3 - 1e-9) and (v[3] - v[2] >= 13.3 - 1e-9)
    print(f"({r7})({sp}): waves spectral {v[0]:.1f}, temporal {v[1]:.1f}; wind spectral {v[2]:.1f}, temporal {v[3]:.1f}%"
          f" | in tolerance {hits}/4 | waves better spectral and wind better temporal by two sounds or more: {part}")
    print("      per category spectral/temporal/st: " + ", ".join(f"{c} {pc['spec'][i]:.0f}/{pc['temp'][i]:.0f}/{pc['st'][i]:.0f}" for i, c in enumerate(CATS)))

**Result: waves are identified by spectral correlations (100%) far better than by temporal ones (33.3 to 40%), and wind by temporal (80%) better than by spectral ones (33.3 to 40%), under every reading; three of four values are within tolerance at best, wind spectral (33.3%) being below 47 +/- 6.7%.**

## fig9-spectrum-classifier-lower

> The spectrum-based classifier does not depend on temporal resolution and performs below all
> three correlation classifiers.

Range of the spectrum classifier's full-duration accuracy over the ten resolutions (under 10
points for "does not depend"), and at every duration of each correlation classifier's curve (at
its optimum) the spectrum classifier at its own optimum, taken at the longest of its durations
not exceeding that duration.

In [ ]:
for r7, sp in READ:
    v = [at10("spectrum", r7, sp, tw) for tw in M.RES_MS]
    tws = OPT[("spectrum", r7, sp)]
    ts = np.array(NS[(sp, tws)]) * SPACING[tws][sp]
    ys = 100 * ACC[("spectrum", r7, sp, tws)].mean(0)
    below, below10 = {}, {}
    for k in ("spec", "temp", "st"):
        tw = OPT[(k, r7, sp)]
        t = np.array(NS[(sp, tw)]) * SPACING[tw][sp]
        y = 100 * ACC[(k, r7, sp, tw)].mean(0)
        yspec = np.array([ys[max(0, np.searchsorted(ts, x + 1e-9) - 1)] for x in t])
        below[k] = (int((yspec < y).sum()), len(t), float((y - yspec).min()))
        below10[k] = ys[-1] < y[-1]
    allb = all(b[0] == b[1] for b in below.values())
    print(f"({r7})({sp}): spectrum classifier at full duration {min(v):.1f} to {max(v):.1f}% over resolutions (range {max(v) - min(v):.1f} points,"
          f" under 10: {max(v) - min(v) < 10}); optimum {tws:.0f} ms, {ys[-1]:.1f}%")
    print("      durations where it is below: " + ", ".join(f"{k} {b[0]}/{b[1]} (smallest margin {b[2]:+.1f})" for k, b in below.items())
          + f" | below at every duration: {allb} | below all three at full duration: {all(below10.values())}")

**Result: the spectrum classifier varies by 9.7 to 10.3 points over resolution, is best at 25 or 50 ms (80.0 to 80.5%), and is not below all three correlation classifiers at the full duration under any reading; it is above the temporal classifier at most durations.**

## fig10-vocalization-background

> In a vocalization against background task, the spectro-temporal classifier approaches 90% for
> background and nearly 100% for vocalizations; vocalization accuracy improves with duration while
> background accuracy stays constant, and vocalizations are classified better with spectral than
> temporal correlations.

Vocalizations: single bird songs, cat meowing, dog barking, human speech (60 sounds); background:
the other nine categories (135). Readings (a) one GMM per class fitted on its pooled sounds (same
PCs and N_c as the 13-category classifier, leave one sound out), (b) the 13-category decision
mapped to its class. Each classifier at its 13-category optimum. Spectral against temporal on the
vocalizations: paired t test over the 60 sounds at each duration, Bonferroni over durations (the
two-sample test is shown too). The temporal minus spectro-temporal difference is read (c) at the
full duration and (d) as the mean over durations.

In [ ]:
VOC = [CATS.index(c) for c in ("Single_Bird_Songs", "Cat_Meowing", "Dog_Barking", "Human_Speech")]
CLS = np.isin(LAB, VOC).astype(int)  # 1 vocalization, 0 background
def two_class(r7, sp):
    res = {}
    for k in ("spec", "temp", "st"):
        tw = OPT[(k, r7, sp)]
        ns = NS[(sp, tw)]
        accb = np.array([[CLS[j] == int(np.isin(np.argmax(np.cumsum(L, 0)[n - 1]), VOC)) for n in ns] for j, L in enumerate(LL[(k, r7, sp, tw)])], float)
        idx = TIDX[tw][sp]
        npc, nc = M.REPORTED[k] if r7 == "7b" else (NPC90[(k, tw)], M.REPORTED[k][1])
        Z = ZS[(k, tw)]
        ll, _, _ = M.loso(Z, CLS, npc, nc, [{"w": z[idx]} for z in Z], SEED["two"])
        acca = M.accuracy_curve(ll["w"], CLS, np.arange(2), ns)
        res[k] = {"a": acca, "b": accb, "t": np.array(ns) * SPACING[tw][sp]}
    return res
TWO = {}
for r7, sp in (("7b", "a"), ("7b", "b")):
    TWO[(r7, sp)] = two_class(r7, sp)
for (r7, sp), res in TWO.items():
    for rd in ("a", "b"):
        st = res["st"][rd]
        voc, bg = 100 * st[CLS == 1].mean(0), 100 * st[CLS == 0].mean(0)
        ok_v = abs(voc[-1] - 100) <= 10 and abs(bg[-1] - 90) <= 9
        rises = voc[-1] > voc[0]
        flat = bg.max() - bg.min() < 10
        n = st.shape[1]
        sv, tv = res["spec"][rd][CLS == 1], res["temp"][rd][CLS == 1]
        m = min(sv.shape[1], tv.shape[1])
        pp = [stats.ttest_rel(sv[:, i], tv[:, i]).pvalue * m if (sv[:, i] != tv[:, i]).any() else 1.0 for i in range(m)]
        p2 = [stats.ttest_ind(sv[:, i], tv[:, i]).pvalue * m for i in range(m)]
        better = [sv[:, i].mean() > tv[:, i].mean() for i in range(m)]
        sig = [better[i] and pp[i] < 0.01 for i in range(m)]
        tm = 100 * res["temp"][rd][CLS == 1].mean(0)
        k_ = min(len(tm), len(voc))
        diff_c, diff_d = tm[-1] - voc[-1], float(np.mean(tm[:k_] - voc[:k_]))
        print(f"({r7})({sp}) reading ({rd}): spectro-temporal at full duration background {bg[-1]:.1f}%, vocalizations {voc[-1]:.1f}% (both in tolerance: {ok_v})"
              f" | vocalizations rise from {voc[0]:.1f}%: {rises} | background range {bg.min():.1f} to {bg.max():.1f} (under 10 points: {flat})")
        print(f"      vocalizations, spectral {100 * sv[:, -1].mean():.1f} against temporal {100 * tv[:, -1].mean():.1f}% at full duration; spectral higher and"
              f" Bonferroni paired p < 0.01 at {sum(sig)} of {m} durations (smallest p {min(pp):.1e}; two-sample smallest {min(p2):.1e})"
              f" | temporal minus spectro-temporal (c) {diff_c:+.1f}, (d) {diff_d:+.1f} points (in -20 +/- 2: {abs(diff_c + 20) <= 2}, {abs(diff_d + 20) <= 2})")
        for k in ("spec", "temp", "st"):
            print(f"      {k:5s} vocalizations " + " ".join(f"{100 * x:.0f}" for x in res[k][rd][CLS == 1].mean(0))
                  + " | background " + " ".join(f"{100 * x:.0f}" for x in res[k][rd][CLS == 0].mean(0)))

**Result: under (6b)(7b) with one model per class, the spectro-temporal classifier is 93.3% on background and 96.7% on vocalizations at the full duration, vocalizations rise from 81.7% while background stays within 87.4 to 94.1%, and the temporal classifier is 20.5 points below the spectro-temporal one on vocalizations averaged over durations; spectral beats temporal on vocalizations at every duration but at Bonferroni p < 0.01 at 1 of 10 durations only. In the (6a)(7b) rows the spectral and temporal curves come from different optima and the per-duration tests pair them by index.**

## Variants: N_c by cross-validated likelihood, and PCA refitted in every fold

Reading (7a) also chooses N_c by cross-validated likelihood (Methods, N_c = 1 to 20). At each
classifier's (7b)(a) optimum, for N_c of 1, 2, 3, 5, 8, 13 and 20 (a subset of the Methods' 1 to
20, for time): 3-fold cross-validation over the sounds of each category (5 sounds held out each
time), the mean held-out log-likelihood per observation averaged over the 13 categories and
folds. Then the spectral classifier at its optimum with the PCA refitted without the held-out
sound, for the first two sounds of each category, to show what fitting it once on all sounds
does.

In [ ]:
t1 = time.time()
NC_GRID = [1, 2, 3, 5, 8, 13, 20]
for k in ("spec", "temp", "st"):
    tw = OPT[(k, "7b", "a")]
    Z = ZS[(k, tw)]
    npc = M.REPORTED[k][0]
    def cv_ll(nc):
        tot = 0.0
        for c in range(13):
            idx = np.flatnonzero(LAB == c)
            for f in range(3):
                te = idx[f::3]; tr = np.setdiff1d(idx, te)
                gm = M.fit_gmm(M.train_rows([Z[i] for i in tr], npc, np.random.default_rng(SEED["nc"] + c * 10 + f), cap=5000), nc)
                X = M.train_rows([Z[i] for i in te], npc, np.random.default_rng(SEED["nc"] + c * 10 + f + 1), cap=5000)
                tot += gm.score(X.astype(np.float64)) / 39
        return tot
    cv = Parallel(n_jobs=4)(delayed(cv_ll)(nc) for nc in NC_GRID)
    best = NC_GRID[int(np.argmax(cv))]
    print(f"{k} at {tw:.0f} ms, {npc} PCs: cross-validated N_c {best} (reported {M.REPORTED[k][1]}); mean held-out log-likelihood per observation: "
          + ", ".join(f"{nc}: {v:.2f}" for nc, v in zip(NC_GRID, cv)), flush=True)
# PCA refitted without the held-out sound, all 13 category models refitted, for two sounds per category
tw = OPT[("spec", "7b", "a")]
Fs = Parallel(n_jobs=4)(delayed(M.features)(c.astype(np.float64), tw) for c in COCH)
idx = TIDX[tw]["a"]
npc, nc = M.REPORTED["spec"]
def fold(j):
    tr = [F for i, F in enumerate(Fs) if i != j]
    mu, co, _ = M.fit_pca(tr, "spec", np.random.default_rng(SEED["fold_pca"] + j))
    Zt = [M.project(F, "spec", mu, co) for F in Fs]
    ll = np.array([M.window_ll(M.fit_gmm(M.train_rows([Zt[i] for i in np.flatnonzero(LAB == c) if i != j], npc,
                                                        np.random.default_rng(j * 13 + c)), nc), Zt[j][idx], npc) for c in range(13)]).T
    return bool(np.cumsum(ll, 0)[-1].argmax() == LAB[j])
FOLDS = [int(i) for c in range(13) for i in np.flatnonzero(LAB == c)[:2]]
hit = Parallel(n_jobs=4)(delayed(fold)(j) for j in FOLDS)
one = ACC[("spec", "7b", "a", tw)][FOLDS, -1]
print(f"spectral at {tw:.0f} ms, (7b)(a), the first two sounds of each category ({len(FOLDS)} folds): PCA refitted in each fold {100 * np.mean(hit):.1f}%,"
      f" one PCA on all sounds {100 * one.mean():.1f}%; the two agree on {int((np.array(hit) == one.astype(bool)).sum())} of {len(FOLDS)} sounds")
del Fs
print(f"variants took {(time.time() - t1) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig9a-optimal-resolution | reproduced under (6b)(7b) (141, 141 and 100 ms) |
| fig9b-similar-maximum | reproduced under the (7b) readings |
| fig9b-accumulation-rate | partially reproduced (order holds, times do not) |
| fig9c-time-averaged-worse | partially reproduced (15 to 37 points lower under reading b, not 25) |
| fig9e-category-specific | partially reproduced (3 of 4 values) |
| fig9-spectrum-classifier-lower | not reproduced |
| fig10-vocalization-background | reproduced under (6b)(7b), reading (a) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the deposit has no classifier code, so the classifier
follows the Methods, with these choices of its own: the spectro-temporal feature vector is the
spectral and temporal features side by side; PCA is fitted once on all sounds (refitting it per
fold changed 1 of 26 tested decisions); training observations are subsampled to 10000 per
category; the spectrum classifier uses the PCs for 90% and 8 components; the time-averaged
classifier of Fig 9C is not described and two readings are run; the cross-validated choice of
N_c was run at the optima only, on a subset of 1 to 20. PCs explaining 90% at 141 ms are 26
(spectral) and 8 (temporal), the paper's counts; the cross-validated N_c of the spectro-temporal
classifier (13 at 141 ms) matches the reported one, those of the spectral (3 at 283 ms) and
temporal (20 at 141 ms) classifiers do not. What was found about the dataset: the 195 model sounds are all there and suffice for the
classifier.

In [ ]:
print(f"total run time {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")